# EDA of the Flicker8k Dataset
**Team Name and Members:**
* Team Name: **demidova_keller** (used in submitted file names)
* Student 1: Iuliia Demidova (iuliia.demidova@students.fhnw.ch)
* Student 2: Lucas Keller (lucas.keller@students.fhnw.ch)

In [ ]:
from pathlib import Path
from collections import Counter
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option("display.max_colwidth", 160)

# Change this if the notebook is not placed in the dataset root.
# Expected structure:
# DATA_DIR/
#   images/
#   splits/
#     train_captions.csv
#     train_images.txt
#     test_captions.csv
#     test_images.txt
#   captions.txt

DATA_DIR = Path("data")
IMAGES_DIR = DATA_DIR / "images"
SPLITS_DIR = DATA_DIR / "splits"

CAPTIONS_PATH = DATA_DIR / "captions.txt"
TRAIN_CAPTIONS_PATH = SPLITS_DIR / "train_captions.csv"
TEST_CAPTIONS_PATH = SPLITS_DIR / "test_captions.csv"
TRAIN_IMAGES_PATH = SPLITS_DIR / "train_images.txt"
TEST_IMAGES_PATH = SPLITS_DIR / "test_images.txt"

RANDOM_SEED = 13
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 1. Explore the tokenizer `tokenzer.py` implementation

In [ ]:
from tokenizer import CaptionTokenizer, SPECIAL_TOKENS, TOKEN_PATTERN, normalize_text, tokenize
print("Imported tokenizer.py from project.")


print("SPECIAL_TOKENS:", SPECIAL_TOKENS)
print("TOKEN_PATTERN:", TOKEN_PATTERN.pattern)

example = "A cat runs, jumps, and catches a bird."
print("Example caption:", example)
print("Tokens:", tokenize(example))


## 2. Load captions and splits

Verify image IDs, captions, and possible leakage between splits.


In [ ]:
def read_image_list(path: Path) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def normalize_caption_df(df: pd.DataFrame) -> pd.DataFrame:
    """Make column names robust across common Flickr8k formats."""
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]

    possible_image_cols = ["image", "image_id", "filename", "file", "img"]
    possible_caption_cols = ["caption", "comment", "sentence", "text"]

    image_col = next((c for c in possible_image_cols if c in df.columns), None)
    caption_col = next((c for c in possible_caption_cols if c in df.columns), None)

    if image_col is None or caption_col is None:
        raise ValueError(f"Could not infer image/caption columns. Found columns: {list(df.columns)}")

    return (
        df[[image_col, caption_col]]
        .rename(columns={image_col: "image", caption_col: "caption"})
        .assign(
            image=lambda x: x["image"].astype(str).str.strip(),
            caption=lambda x: x["caption"].astype(str).str.strip(),
        )
    )


captions = normalize_caption_df(pd.read_csv(CAPTIONS_PATH))
train_captions = normalize_caption_df(pd.read_csv(TRAIN_CAPTIONS_PATH))
test_captions = normalize_caption_df(pd.read_csv(TEST_CAPTIONS_PATH))

train_images = read_image_list(TRAIN_IMAGES_PATH)
test_images = read_image_list(TEST_IMAGES_PATH)

print("Loaded:")
print(f"  captions.txt:        {len(captions):>6} caption rows, {captions['image'].nunique():>5} unique images")
print(f"  train_captions.csv:  {len(train_captions):>6} caption rows, {train_captions['image'].nunique():>5} unique images")
print(f"  test_captions.csv:   {len(test_captions):>6} caption rows, {test_captions['image'].nunique():>5} unique images")
print(f"  train_images.txt:    {len(train_images):>6} image names")
print(f"  test_images.txt:     {len(test_images):>6} image names")
print(f"Train ratio:    {len(train_images)/(len(train_images)+len(test_images))*100:>6.2f}% ")
print(f"Test ratio:    {len(test_images)/(len(train_images)+len(test_images))*100:>6.2f}% ")


captions.head()


## 3. Dataset integrity checks

In [ ]:
train_image_set = set(train_images)
test_image_set = set(test_images)

summary = pd.DataFrame([
    {
        "split": "full",
        "caption_rows": len(captions),
        "unique_images_in_captions": captions["image"].nunique(),
        "images_in_txt": None,
        "avg_captions_per_image": len(captions) / captions["image"].nunique(),
    },
    {
        "split": "train",
        "caption_rows": len(train_captions),
        "unique_images_in_captions": train_captions["image"].nunique(),
        "images_in_txt": len(train_images),
        "avg_captions_per_image": len(train_captions) / train_captions["image"].nunique(),
    },
    {
        "split": "test",
        "caption_rows": len(test_captions),
        "unique_images_in_captions": test_captions["image"].nunique(),
        "images_in_txt": len(test_images),
        "avg_captions_per_image": len(test_captions) / test_captions["image"].nunique(),
    },
])

display(summary)

leaked_images = train_image_set & test_image_set
missing_train_captions = train_image_set - set(train_captions["image"])
missing_test_captions = test_image_set - set(test_captions["image"])
extra_train_caption_images = set(train_captions["image"]) - train_image_set
extra_test_caption_images = set(test_captions["image"]) - test_image_set

print(f"Train/test image leakage: {len(leaked_images)}")
print(f"Train images without captions: {len(missing_train_captions)}")
print(f"Test images without captions: {len(missing_test_captions)}")
print(f"Train caption images not in train_images.txt: {len(extra_train_caption_images)}")
print(f"Test caption images not in test_images.txt: {len(extra_test_caption_images)}")

caption_counts = captions.groupby("image").size()
caption_counts.describe()


$\rightarrow$ Every image has exactly 5 captions, which is consistent with the standard Flickr8k dataset. No leakage between train/test. No missing captions for images in the splits.

## 4. Random samples exploration

In [ ]:
def show_image_with_captions(image_name: str, df: pd.DataFrame = captions):
    image_path = IMAGES_DIR / image_name
    refs = df.loc[df["image"] == image_name, "caption"].tolist()

    if not image_path.exists():
        print(f"Missing image file: {image_path}")
        print("Captions:")
        for i, cap in enumerate(refs, 1):
            print(f"{i}. {cap}")
        return

    img = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    print(image_name)
    for i, cap in enumerate(refs, 1):
        print(f"{i}. {cap}")


available_caption_images = sorted(set(captions["image"]))
sample_image = random.choice(available_caption_images)
show_image_with_captions(sample_image)


## 5. Caption length distribution

**The provided tokenizer removes punctuation**, so the given lengths are **word-like token counts**.

For training with `CaptionTokenizer.encode(add_special_tokens=True)`, the full encoded sequence length is:

> _encoded_len = caption_token_len + 2_

because `<bos>` and `<eos>` are added.


In [ ]:
for df in [captions, train_captions, test_captions]:
    df["tokens"] = df["caption"].apply(tokenize)
    df["caption_len"] = df["tokens"].apply(len)
    df["encoded_len"] = df["caption_len"] + 2  # <bos> + <eos>

length_summary = pd.DataFrame({
    "full_caption_tokens": captions["caption_len"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]),
    "train_caption_tokens": train_captions["caption_len"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]),
    "test_caption_tokens": test_captions["caption_len"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]),
    "train_encoded_len": train_captions["encoded_len"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]),
}).round(2)

display(length_summary)

plt.figure(figsize=(8, 4))
plt.hist(train_captions["caption_len"], bins=30, alpha=0.7, label="train")
plt.hist(test_captions["caption_len"], bins=30, alpha=0.7, label="test")
plt.xlabel("Caption length, tokenizer tokens")
plt.ylabel("Number of captions")
plt.title("Caption length distribution")
plt.grid()
plt.legend()
plt.show()

suggested_caption_len = int(np.ceil(train_captions["caption_len"].quantile(0.95)))
suggested_encoded_max_len = suggested_caption_len + 2

print(f"Suggested caption-token max around 95th percentile: {suggested_caption_len}")
print(f"Suggested encoded MAX_LEN for CaptionTokenizer.encode(...): {suggested_encoded_max_len}")


## 6. Vocabulary analysis

Exploring the behavior of `CaptionTokenizer`:

- vocabulary is built **only from training captions**
- tokens with `frequency < min_freq` become `<unk>`
- special tokens are always included first
- remaining words are sorted alphabetically


In [ ]:
train_token_counter = Counter(token for toks in train_captions["tokens"] for token in toks)
test_tokens = [token for toks in test_captions["tokens"] for token in toks]

vocab_rows = []
for min_freq in [1, 2, 3, 4, 5, 6, 10]:
    tokenizer = CaptionTokenizer.from_captions(
        train_captions["caption"],
        min_freq=min_freq,
        special_tokens=SPECIAL_TOKENS,
    )

    test_oov = sum(token not in tokenizer.stoi for token in test_tokens)

    vocab_rows.append({
        "min_freq": min_freq,
        "vocab_size_without_specials": tokenizer.vocab_size - len(SPECIAL_TOKENS),
        "vocab_size_with_specials": tokenizer.vocab_size,
        "test_oov_tokens": test_oov,
        "test_oov_rate": test_oov / max(1, len(test_tokens)),
    })

vocab_df = pd.DataFrame(vocab_rows)
display(vocab_df)


## 9. Image size / aspect ratio check

! For ResNet-based encoders, images are usually resized and normalized with ImageNet statistics.!

This check helps decide whether cropping might remove important content.


In [ ]:
def collect_image_metadata(image_names: list[str]) -> pd.DataFrame:
    rows = []

    for image_name in image_names:
        path = IMAGES_DIR / image_name

        row = {
            "image": image_name,
            "exists": False,
            "width": np.nan,
            "height": np.nan,
            "aspect_ratio": np.nan,
        }

        if not path.exists():
            rows.append(row)
            continue

        try:
            with Image.open(path) as img:
                w, h = img.size

            row.update({
                "exists": True,
                "width": w,
                "height": h,
                "aspect_ratio": w / h,
            })

        except Exception as exc:
            row["error"] = str(exc)

        rows.append(row)

    return pd.DataFrame(rows)


# All images referenced by captions.txt
all_image_names = sorted(captions["image"].unique())

image_meta = collect_image_metadata(all_image_names)

display(image_meta.describe(include="all", percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

missing_images = image_meta["exists"].eq(False).sum()
print(f"Missing/unreadable images: {missing_images}/{len(image_meta)}")

valid_image_meta = image_meta[image_meta["exists"]].copy()

if not valid_image_meta.empty:
    plt.figure(figsize=(8, 4))
    plt.hist(valid_image_meta["aspect_ratio"], bins=60)

    plt.xlabel("Aspect ratio: width / height")
    plt.ylabel("Number of images")
    plt.title("Image aspect ratio distribution")
    plt.grid()
    plt.tight_layout()
    plt.show()
else:
    print("No readable images found. Check IMAGES_DIR.")

$\rightarrow$ Images are mostly non-square, the majority of images are landscape-oriented (aspect ratio > 1).